<a href="https://colab.research.google.com/github/Laraib175/Flyrank-week2/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laraib175/Flyrank-week2/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My Lane as an ML Task

**Task Type:** Ranking / Scoring (supervised, but output is a ranked list)

I am framing this as a **ranking** problem rather than pure classification. The editor does not need a hard "declining / not declining" label — they need an ordered queue: "Which page should I review first?" The model's output is a continuous priority score, and the action is: work top-down until capacity runs out.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or Proxy

**Proxy:** A binary flag `is_opportunity` built from **observed future behavior**, not from an existing derived column.

Definition sketch: A page is a positive (1) if, in the 30 days **after** the feature window, its `gsc_clicks` drops by more than X% relative to the prior 30 days, **or** if its CTR in the feature window is below the 25th percentile for its position tier among pages with meaningful impressions.

**Why not `is_declining_label`:** That column is derived from `trend_pct`, which is in the same row as the features. Using it as the target means the model learns the label formula, not a real-world pattern. The target must come from a later time window than the features.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success Metric

**Metric:** Precision@50

**Why:** The editor's weekly review capacity is roughly 50 pages. The cost of a false positive is a wasted editor hour; the cost of a false negative is a missed page that could have been fixed. Precision@50 directly measures: among the 50 pages we told the editor to fix first, how many were actually worth fixing? Accuracy over 30,000 rows is the wrong question.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")

# 证明一行 = 一个 content item
assert df["content_id"].is_unique, "content_id is not unique!"
print("content_id is unique. One row = one content item.")

# 展示核心列
cols_to_show = ["content_id", "client_id", "content_type", "ctr",
                "avg_position", "impressions_90d", "gsc_clicks"]
df[cols_to_show].head(10)

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/content_refresh_anonymized.csv'

## 4. Unit of Analysis

**One row = one pseudonymized content page.** Verified via `content_id.is_unique` returning True.

This matters because: if I later aggregate to client level or query level, the meaning of the score changes. The editor acts on **pages**, so the unit must be the page.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML Beats a Fixed Rule

**A plain rule is not enough because:**

1. **A flat CTR threshold is too blunt.** A page ranking #2 with 0.3% CTR is a disaster; a page ranking #40 with 0.3% CTR may be normal. Position tier must be adjusted for, and the adjustment factor is not a single constant.

2. **Signals are tangled.** CTR, engagement, position, content type, and traffic volume interact non-linearly. A hand-written if-statement either misses opportunities or floods the queue.

3. **The cost structure demands ranking, not binary flagging.** A rule can only say "flag/not flag". The editor needs "first / second / third...". A scoring model produces a continuous ordering that a threshold rule cannot.

**What I will claim:** decision-support ranking, not causal claims. A high score means "worth reviewing first", not "fixing this will increase traffic".

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.